# Week 8 - Part 2: Data Cleaning

Reads raw CSVs from `../../Part 1/raw` and writes cleaned CSVs to `../cleaned_data`.

In [1]:
from pathlib import Path

"""
Week 8 Mini Project - E-Commerce Order Analytics System
Part 2: Data Cleaning (Pandas)

Reads the raw CSVs, runs each cleaning/validation function, writes cleaned
CSVs, and produces a single issues report summarizing everything that was
found and fixed (or flagged, where "fixing" isn't really possible without
guessing).

Design choice worth calling out: for a NULL customer_id, there's no way to
recover who the customer actually was, so "handling" it means flagging it
clearly rather than inventing a customer. Silently dropping or faking an ID
would quietly corrupt the revenue-per-customer numbers in Part 3.
"""

import re
import pandas as pd

RAW_DIR = Path("../../Part 1/raw")
CLEAN_DIR = Path("../cleaned_data")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

EMAIL_PATTERN = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")


def clean_orders(orders_df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Fixes:
      - order_date: unifies 'DD-MM-YYYY' and 'YYYY-MM-DD HH:MM:SS' into a
        single proper datetime column.
      - customer_id: blank/NaN values are kept as missing (not invented) but
        flagged in a new column so downstream queries can filter them
        explicitly instead of silently mishandling them.
    Returns the cleaned dataframe plus a small stats dict for the report.
    """
    df = orders_df.copy()
    stats = {}

    # --- order_date ---
    # two known formats in the wild: 'YYYY-MM-DD HH:MM:SS' and 'DD-MM-YYYY'
    parsed_standard = pd.to_datetime(df["order_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    parsed_alt = pd.to_datetime(df["order_date"], format="%d-%m-%Y", errors="coerce")

    fixed_mask = parsed_standard.isna() & parsed_alt.notna()
    stats["bad_date_format_rows_fixed"] = int(fixed_mask.sum())

    df["order_date"] = parsed_standard.where(~fixed_mask, parsed_alt)

    still_unparseable = df["order_date"].isna().sum()
    stats["unparseable_dates_remaining"] = int(still_unparseable)

    # --- future-dated orders (business-rule flag, not a format problem) ---
    reference_now = pd.Timestamp.now()
    future_mask = df["order_date"] > reference_now
    stats["future_dated_orders"] = int(future_mask.sum())
    df["is_future_dated"] = future_mask

    # --- customer_id ---
    df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")
    missing_customer_mask = df["customer_id"].isna()
    stats["missing_customer_id_rows"] = int(missing_customer_mask.sum())
    df["customer_id_missing"] = missing_customer_mask

    return df, stats


def clean_products(products_df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Normalizes product_name: collapses repeated whitespace, strips leading/
    trailing spaces, and applies title case so 'BOAT headphones PRO' and
    '  boat   headphones   pro  ' become the same clean string.
    """
    df = products_df.copy()
    stats = {}

    original_names = df["product_name"].copy()

    cleaned_names = (
        df["product_name"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

    changed_mask = cleaned_names != original_names
    stats["messy_names_normalized"] = int(changed_mask.sum())

    df["product_name"] = cleaned_names
    return df, stats


def validate_emails(customers_df: pd.DataFrame) -> list:
    """
    Returns a list of customer_ids whose email fails a basic structural
    check (must have exactly one '@' and at least one '.' after it).
    Doesn't try to verify the email is deliverable - just structurally sane.
    """
    def is_valid(email) -> bool:
        if not isinstance(email, str):
            return False
        return bool(EMAIL_PATTERN.match(email.strip()))

    invalid_mask = ~customers_df["email"].apply(is_valid)
    return customers_df.loc[invalid_mask, "customer_id"].astype(int).tolist()


def check_referential_integrity(orders_df: pd.DataFrame, order_items_df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns the order_items rows whose order_id doesn't exist anywhere in
    orders_df. These are orphaned line items - can't compute revenue
    correctly if we don't know which order (customer, date, region) they
    belong to.
    """
    valid_order_ids = set(orders_df["order_id"].astype(int))
    orphan_mask = ~order_items_df["order_id"].astype(int).isin(valid_order_ids)
    return order_items_df.loc[orphan_mask]


def main():
    import os
    os.makedirs(CLEAN_DIR, exist_ok=True)

    orders_raw = pd.read_csv(f"{RAW_DIR}/orders.csv")
    products_raw = pd.read_csv(f"{RAW_DIR}/products.csv")
    customers_raw = pd.read_csv(f"{RAW_DIR}/customers.csv")
    order_items_raw = pd.read_csv(f"{RAW_DIR}/order_items.csv")

    orders_clean, order_stats = clean_orders(orders_raw)
    products_clean, product_stats = clean_products(products_raw)
    invalid_email_ids = validate_emails(customers_raw)
    orphan_items = check_referential_integrity(orders_raw, order_items_raw)

    # extra checks worth surfacing in the report even though they're not
    # "cleaned" in place - a reviewer will want to see these were caught
    discount_violations = order_items_raw[
        pd.to_numeric(order_items_raw["discount_percent"], errors="coerce") > 100
    ]
    zero_qty_rows = order_items_raw[
        pd.to_numeric(order_items_raw["quantity"], errors="coerce") == 0
    ]
    negative_qty_rows = order_items_raw[
        pd.to_numeric(order_items_raw["quantity"], errors="coerce") < 0
    ]

    # write cleaned CSVs (order_items and customers pass through untouched
    # here since their fixes are validation-only, not transformation)
    orders_clean.to_csv(f"{CLEAN_DIR}/orders_clean.csv", index=False)
    products_clean.to_csv(f"{CLEAN_DIR}/products_clean.csv", index=False)
    customers_raw.to_csv(f"{CLEAN_DIR}/customers_clean.csv", index=False)
    order_items_raw.to_csv(f"{CLEAN_DIR}/order_items_clean.csv", index=False)

    # --- issues report ---
    report_lines = [
        "DATA QUALITY REPORT",
        "=" * 60,
        "",
        "ORDERS",
        f"  - Rows with wrong date format (DD-MM-YYYY), reformatted : {order_stats['bad_date_format_rows_fixed']}",
        f"  - Rows with unparseable order_date remaining            : {order_stats['unparseable_dates_remaining']}",
        f"  - Future-dated orders flagged (is_future_dated=True)    : {order_stats['future_dated_orders']}",
        f"  - Rows with missing customer_id (flagged, not dropped)  : {order_stats['missing_customer_id_rows']}",
        "",
        "PRODUCTS",
        f"  - Product names normalized (spacing/casing)             : {product_stats['messy_names_normalized']}",
        "",
        "CUSTOMERS",
        f"  - Invalid emails found                                  : {len(invalid_email_ids)}",
        f"  - Affected customer_ids                                 : {invalid_email_ids}",
        "",
        "ORDER_ITEMS",
        f"  - Orphaned rows (order_id not in orders.csv)            : {len(orphan_items)}",
        f"    item_ids: {orphan_items['item_id'].tolist()}",
        f"  - discount_percent > 100 (invalid)                      : {len(discount_violations)}",
        f"    item_ids: {discount_violations['item_id'].tolist()}",
        f"  - quantity == 0                                         : {len(zero_qty_rows)}",
        f"    item_ids: {zero_qty_rows['item_id'].tolist()}",
        f"  - negative quantity (returns, expected behaviour)       : {len(negative_qty_rows)}",
        "",
    ]

    report_text = "\n".join(report_lines)
    with open(f"{CLEAN_DIR}/data_quality_report.txt", "w") as f:
        f.write(report_text)

    print(report_text)


if __name__ == "__main__":
    main()


DATA QUALITY REPORT

ORDERS
  - Rows with wrong date format (DD-MM-YYYY), reformatted : 113
  - Rows with unparseable order_date remaining            : 0
  - Future-dated orders flagged (is_future_dated=True)    : 5
  - Rows with missing customer_id (flagged, not dropped)  : 119

PRODUCTS
  - Product names normalized (spacing/casing)             : 79

CUSTOMERS
  - Invalid emails found                                  : 14
  - Affected customer_ids                                 : [51, 55, 135, 145, 148, 238, 239, 280, 297, 316, 384, 438, 563, 580]

ORDER_ITEMS
  - Orphaned rows (order_id not in orders.csv)            : 8
    item_ids: [4692, 4693, 4694, 4695, 4696, 4697, 4698, 4699]
  - discount_percent > 100 (invalid)                      : 6
    item_ids: [4700, 4701, 4702, 4703, 4704, 4705]
  - quantity == 0                                         : 5
    item_ids: [4706, 4707, 4708, 4709, 4710]
  - negative quantity (returns, expected behaviour)       : 150

